# Inspect a Polymerase tree




In [1]:
from __future__ import annotations

from collections import deque
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Code, Markdown, display

def _find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'conf').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root from current working directory.')

BASE_DEFT = _find_repo_root()

if str(BASE_DEFT) not in sys.path:
    sys.path.insert(0, str(BASE_DEFT))

from src.data.polymerase_loader import load_polymerase
from src.utils.lightweight_tree import load_pickled_tree
from src.utils.tree import feature_output_to_series


In [2]:
TREE_PATH = BASE_DEFT / 'artifacts/trees/polymerase/deft_lightweight/tree_001_seed42.dill'

artifact = load_pickled_tree(TREE_PATH)
root = artifact.root if hasattr(artifact, 'root') else artifact

top_level_fields = pd.DataFrame({'field': sorted(vars(artifact).keys())})
node_fields = pd.DataFrame({'field': sorted(vars(root).keys())})
feature_fields = pd.DataFrame({'field': sorted(vars(root.feature).keys())})

metadata_df = pd.DataFrame(
    [
        {
            'tree_path': str(TREE_PATH),
            'dataset_name': getattr(artifact, 'dataset_name', None),
            'method_name': getattr(artifact, 'method_name', None),
            'random_seed': getattr(artifact, 'random_seed', None),
            'max_depth': getattr(artifact, 'max_depth', None),
            'source_tree_path': getattr(artifact, 'source_tree_path', None),
            'source_file_size_bytes': getattr(artifact, 'source_file_size_bytes', None),
            'lightweight_file_size_bytes': TREE_PATH.stat().st_size,
            'format_version': getattr(artifact, 'format_version', None),
        }
    ]
)

root_summary_df = pd.DataFrame(
    [
        {
            'feature_name': root.feature.name,
            'threshold': root.feature.threshold,
            'prediction_at_root': root.prediction,
            'description': root.feature.description,
        }
    ]
)

display(Markdown('## Lightweight Artifact Metadata'))
display(metadata_df)

display(Markdown('## Fields Stored In The Lightweight Objects'))
display(Markdown('### Top-Level Artifact Fields'))
display(top_level_fields)
display(Markdown('### Node Fields'))
display(node_fields)
display(Markdown('### Feature Fields'))
display(feature_fields)

display(Markdown('## Root Node Summary'))
display(root_summary_df)

display(Markdown('## Root Feature Code'))
display(Code(root.feature.string, language='python'))


## Lightweight Artifact Metadata

,tree_path,dataset_name,method_name,random_seed,max_depth,source_tree_path,source_file_size_bytes,lightweight_file_size_bytes,format_version
0,/home/nvth2/deft_cr/artifacts/trees/polymerase...,polymerase,deft,42,10,artifacts/trees/polymerase/deft/tree_001_seed4...,108724327,30019,lightweight_tree_v1


## Fields Stored In The Lightweight Objects

### Top-Level Artifact Fields

,field
0,dataset_name
1,exported_at_utc
2,format_version
3,max_depth
4,method_name
5,random_seed
6,root
7,source_file_size_bytes
8,source_tree_path


...

,field
0,description
1,fn
2,name
3,string
4,threshold


## Root Node Summary

,feature_name,threshold,prediction_at_root,description
0,upstream_G_content_20_49,0.25,0.50775,Calculate the proportion of guanine (G) nucleo...


## Root Feature Code

def calculate_upstream_G_content_20_49(X):
    return X['raw_sequence'].apply(lambda seq: seq[20:50].count('G') / 30)

In [3]:
def walk_tree(root):
    rows = []
    queue = deque([('', 0, root)])

    while queue:
        path, depth, node = queue.popleft()
        feature = node.feature
        rows.append(
            {
                'path': path or 'root',
                'depth': depth,
                'is_leaf': node._is_leaf(),
                'prediction': node.prediction,
                'feature_name': getattr(feature, 'name', None),
                'threshold': getattr(feature, 'threshold', None),
                'description': getattr(feature, 'description', None),
                'has_code_string': bool(getattr(feature, 'string', None)),
            }
        )

        if node.left is not None:
            queue.append((path + 'L', depth + 1, node.left))
        if node.right is not None:
            queue.append((path + 'R', depth + 1, node.right))

    return pd.DataFrame(rows).sort_values(['depth', 'path']).reset_index(drop=True)


def format_condition(feature_name: str, threshold: float, direction: str) -> str:
    op = '<=' if direction == 'L' else '>'
    return f'{feature_name} {op} {threshold:.6g}'


def collect_leaf_rules(node, path='', rules=None):
    if rules is None:
        rules = []

    if node._is_leaf():
        return [
            {
                'leaf_path': path or 'root',
                'depth': len(path),
                'prediction': node.prediction,
                'rule': ' and '.join(rules) if rules else '(all samples)',
            }
        ]

    feature = node.feature
    left_rule = format_condition(feature.name, feature.threshold, 'L')
    right_rule = format_condition(feature.name, feature.threshold, 'R')

    return (
        collect_leaf_rules(node.left, path + 'L', rules + [left_rule])
        + collect_leaf_rules(node.right, path + 'R', rules + [right_rule])
    )


def render_tree_text(node, path='', depth=0, max_depth=3):
    label = path or 'root'
    prediction = 'nan' if node.prediction is None else f'{node.prediction:.4f}'
    indent = '  ' * depth

    if node._is_leaf():
        return [f'{indent}{label}: leaf(pred={prediction})']

    if depth >= max_depth:
        return [
            f'{indent}{label}: split(feature={node.feature.name}, threshold={node.feature.threshold:.6g}, pred={prediction})'
        ]

    lines = [
        f'{indent}{label}: if {node.feature.name} <= {node.feature.threshold:.6g} (pred={prediction})'
    ]
    lines.extend(render_tree_text(node.left, path + 'L', depth + 1, max_depth=max_depth))
    lines.append(f'{indent}else:')
    lines.extend(render_tree_text(node.right, path + 'R', depth + 1, max_depth=max_depth))
    return lines


nodes_df = walk_tree(root)
leaf_rules_df = pd.DataFrame(collect_leaf_rules(root)).sort_values(['depth', 'leaf_path']).reset_index(drop=True)

tree_summary_df = pd.DataFrame(
    [
        {
            'node_count': len(nodes_df),
            'leaf_count': int(nodes_df['is_leaf'].sum()),
            'internal_node_count': int((~nodes_df['is_leaf']).sum()),
            'max_depth_seen': int(nodes_df['depth'].max()),
        }
    ]
)

display(Markdown('## Tree Summary'))
display(tree_summary_df)

display(Markdown('## Tree Sketch (first 3 split levels)'))
display(Markdown('```text\n' + '\n'.join(render_tree_text(root, max_depth=2)) + '\n```'))


## Tree Summary

,node_count,leaf_count,internal_node_count,max_depth_seen
0,97,49,48,10


## Tree Sketch (first 3 split levels)

```text
root: if upstream_G_content_20_49 <= 0.25 (pred=0.5078)
  L: if pos_50_is_G_and_pos_51_is_T <= 0.5 (pred=0.1843)
    LL: split(feature=total_AT_content_0_100, threshold=0.430693, pred=0.0851)
  else:
    LR: split(feature=upstream_base_pair_stacking_energy_10_49, threshold=-1.77103, pred=0.7171)
else:
  R: if pos_50_purine_51_pyrimidine <= 0.5 (pred=0.7334)
    RL: split(feature=pos_51_pyrimidine, threshold=0.5, pred=0.3315)
  else:
    RR: split(feature=central_GC_content_49_51, threshold=0.5, pred=0.9160)
```